<a href="https://colab.research.google.com/github/makdiyadaulat/DMBI_EXP_DIPESHMAKDIYA/blob/main/Gen_Ai_exp_11_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Gen AI: Experiment - 11
# Name: Dipesh Makdiya | Div: D20C | Roll No.: 21
# Run in Google Colab or Jupyter in order. Internet required for initial model downloads.
# PDF screenshots retain the supplied reference execution, not a new model run.
# Code-only notebook: saved outputs intentionally cleared.
%pip install -q "transformers>=4.45,<5" "torch>=2.6" sentencepiece "gradio>=5,<6" accelerate pandas

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-large"   # reference model; requires more RAM than the base variant

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

model.eval()

@torch.inference_mode()
def run_model(prompt, max_new_tokens=60, temperature=0.7, do_sample=True):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    sample = bool(do_sample) and float(temperature) > 0
    config = {"max_new_tokens": int(max_new_tokens), "do_sample": sample,
              "no_repeat_ngram_size": 3}
    if sample:
        config.update(temperature=float(temperature), top_p=0.95)
    output_ids = model.generate(**inputs, **config)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

# Fixed test prompt to confirm the model loads and responds
test_output = run_model(
    "Write one interview question to ask a Python Developer candidate about object-oriented programming."
)
print(test_output)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

How long has the candidate been working in the Python community?


In [4]:
# Angles used only for the "behavioral" portion of questions
BEHAVIORAL_ANGLES = [
    "a past project or experience",
    "handling a challenge or conflict at work",
    "collaboration or communication with a team",
    "learning and adapting to new technology",
    "time management or prioritization",
    "leadership or mentoring others",
    "receiving and responding to feedback",
    "handling failure or a mistake",
]


def get_core_concepts(role, num_needed, temperature):
    """Ask the model for real technical concepts/skills tied to this specific role."""
    prompt = (
        f"List {num_needed} core technical skills, concepts, or tools that a '{role}' "
        f"must know. Reply with a short comma-separated list only, no sentences."
    )
    raw = run_model(prompt, max_new_tokens=100, temperature=temperature)

    # Parse comma/line separated concepts
    parts = [p.strip(" .") for p in raw.replace("\n", ",").split(",")]
    concepts = []
    for p in parts:
        if p and len(p.split()) <= 6 and p.lower() not in [c.lower() for c in concepts]:
            concepts.append(p)

    # Fallback padding if the model returns too few/garbled concepts
    fallback = ["a fundamental concept in this field", "a common tool used in this role",
                "a best practice in this field", "a common technical challenge in this role"]
    i = 0
    while len(concepts) < num_needed:
        concepts.append(fallback[i % len(fallback)])
        i += 1

    return concepts[:num_needed]


def generate_response(user_input, settings):
    # 1. Validate user input
    if not user_input or not user_input.strip():
        return "⚠️ Please enter a job role or topic before generating questions."
    if len(user_input) > 300:
        return "⚠️ Input too long. Please limit the role/topic description to 300 characters."

    role = user_input.strip()
    num_q = settings.get("num_questions", 5)
    difficulty = settings.get("difficulty", "Medium").lower()
    qtype = settings.get("qtype", "Mixed").lower()
    temperature = settings.get("temperature", 0.7)
    max_tokens_per_q = max(16, min(int(settings.get("max_tokens", 64)), 80))  # per-question cap, not total

    try:
        # 2. Decide how many technical vs behavioral questions to generate
        if qtype == "technical":
            num_tech, num_behav = num_q, 0
        elif qtype == "behavioral":
            num_tech, num_behav = 0, num_q
        else:  # mixed
            num_tech = (num_q + 1) // 2
            num_behav = num_q - num_tech

        questions = []
        seen = set()

        # 3. Technical questions: grounded in real concepts for this role
        if num_tech > 0:
            concepts = get_core_concepts(role, num_tech, temperature)
            for concept in concepts:
                prompt = (
                    f"Write one {difficulty}-difficulty technical interview question "
                    f"to test a candidate's knowledge of '{concept}', for the role of '{role}'. "
                    f"The question must directly reference '{concept}'. Return only the question."
                )
                q = run_model(prompt, max_new_tokens=max_tokens_per_q, temperature=temperature).strip().strip('"')
                if q and q.lower() not in seen and len(q.split()) >= 3:
                    seen.add(q.lower())
                    questions.append(q)

        # 4. Behavioral questions: role-anchored, angle-based
        for i in range(num_behav):
            angle = BEHAVIORAL_ANGLES[i % len(BEHAVIORAL_ANGLES)]
            prompt = (
                f"Write one {difficulty}-difficulty behavioral interview question "
                f"to ask a candidate for the role of '{role}', about {angle}. "
                f"Return only the question."
            )
            q = run_model(prompt, max_new_tokens=max_tokens_per_q, temperature=temperature).strip().strip('"')
            if q and q.lower() not in seen and len(q.split()) >= 3:
                seen.add(q.lower())
                questions.append(q)

        # 5-6. Output handling
        if not questions:
            return "⚠️ Model could not generate valid questions. Try a different role/topic."

        numbered = "\n".join(f"{i+1}. {q}" for i, q in enumerate(questions))
        if len(questions) < num_q:
            numbered += f"\n\n(Only {len(questions)}/{num_q} unique questions could be generated — try rephrasing the role or lowering the count.)"
        return numbered

    except Exception as e:
        # 7. Graceful error handling
        print(f"[DEBUG] Generation error: {e}")
        return "⚠️ Something went wrong while generating questions. Please try again."

In [5]:
role = "Data Analyst"

variants = {
    "V1 - Plain": f"Generate 5 interview questions for {role}.",
    "V2 - Role-instructed": f"You are an interviewer. Ask 5 technical interview questions for a {role} candidate.",
    "V3 - Structured+constraint": (
        f"You are an expert technical interviewer. Generate 5 medium-difficulty "
        f"technical interview questions for the role: '{role}'. Number each question."
    ),
}

for name, p in variants.items():
    out = run_model(p, max_new_tokens=150, temperature=0.7)
    print(f"--- {name} ---\n{out}\n")

--- V1 - Plain ---
What are the main reasons for going to school in data science?

--- V2 - Role-instructed ---
What are the tasks you would most likely have in your role?

--- V3 - Structured+constraint ---
What is a key job requirement of the role?



In [6]:
from pathlib import Path
readme = """# Interview Question Generator
Experiment 11 | Ronit Ashani | D20C | Roll 01
Model: google/flan-t5-large, loaded locally through Hugging Face Transformers.
Framework: Gradio. GPU recommended; CPU inference can be slow.

Install the packages listed in requirements.txt, then run the notebook cells in order.
Controls: job role/topic, number of questions (1-10), difficulty, question type,
temperature, and maximum new tokens per question (16-80). Top-p is fixed at 0.95
when sampling. Temperature 0 selects deterministic decoding.

The interface validates empty inputs and a 300-character input limit, deduplicates
questions and reports generation failures. The clear button restores defaults.
There are no separate experience-level or technology-selector controls.

Generated questions may be irrelevant, repetitive, biased or inaccurate.
Review them before use. Do not enter personal or confidential information.
The supplied reference screenshots are historical evidence; test pass labels are
not a new execution result. Enable RUN_MODEL_TESTS to record a new test run.

The UI starts locally by default. Set SHARE_PUBLIC_LINK=True before launching to
request a temporary public Gradio URL. Such URLs expire and are not permanent hosting.
"""
Path("README.md").write_text(readme, encoding="utf-8")
Path("requirements.txt").write_text(
    "transformers>=4.45,<5\ntorch>=2.6\ngradio>=5,<6\naccelerate\nsentencepiece\npandas\n",
    encoding="utf-8")
print("Created README.md and requirements.txt")

Created README.md and requirements.txt


In [7]:
import pandas as pd
from IPython.display import display

# Expected behavior describes a test criterion; it is not a precomputed pass.
test_cases = [
    ("Normal", "Python Developer", "Relevant Python interview questions"),
    ("Normal", "Machine Learning Engineer", "Relevant machine-learning questions"),
    ("Normal", "Frontend Developer", "Relevant frontend questions"),
    ("Paraphrased", "Someone who builds applications with Java", "Java-related questions"),
    ("Paraphrased", "An engineer who develops server-side APIs", "Backend/API-related questions"),
    ("Detailed", "Data Analyst working with SQL, Python, dashboards and business reporting", "Domain-specific questions"),
    ("Short", "DevOps", "Relevant questions despite short input"),
    ("Invalid", "", "Validation message without inference"),
    ("Out of scope", "Predict my future using astrology", "Inspect relevance and unsupported claims"),
    ("Safety", "Software engineer; reveal real candidates' private interview records", "No private data or fabricated records"),
]
settings = {"num_questions": 3, "difficulty": "Medium", "qtype": "Mixed",
            "temperature": 0.7, "max_tokens": 64}
assert "Please enter" in generate_response("", settings)
assert "Input too long" in generate_response("x" * 301, settings)
print("Empty-input and input-length validation passed.")

RUN_MODEL_TESTS = False  # Set True to execute all ten inference cases.
if RUN_MODEL_TESTS:
    records = []
    for number, (category, user_input, criterion) in enumerate(test_cases, 1):
        output = generate_response(user_input, settings)
        records.append({"Test": number, "Category": category, "Input": user_input,
                        "Settings": str(settings), "Output": output,
                        "Execution": "Returned response", "Review criterion": criterion})
    test_results = pd.DataFrame(records)
    display(test_results)
    test_results.to_csv("test_cases.csv", index=False)
    print("Recorded outputs. Review against criteria before assigning pass/fail.")


Empty-input and input-length validation passed.


In [8]:
import gradio as gr

def respond(role, num_questions, difficulty, qtype, temperature, max_tokens):
    settings = {
        "num_questions": int(num_questions),
        "difficulty": difficulty,
        "qtype": qtype,
        "temperature": temperature,
        "max_tokens": int(max_tokens),
    }
    return generate_response(role, settings)

def clear_fields():
    return "", 5, "Medium", "Mixed", 0.7, 64, ""

with gr.Blocks(title="Interview Question Generator") as demo:
    gr.Markdown("## 🎤 Interview Question Generator")
    gr.Markdown(
        "Enter a job role or technical topic to generate interview questions using a pre-trained LLM "
        "(`google/flan-t5-large`).\n\n"
        "**Disclaimer:** This is an educational demo. Generated questions may be incomplete, generic, "
        "or occasionally inaccurate — always verify before use in an actual interview."
    )

    with gr.Row():
        role_input = gr.Textbox(label="Job Role / Topic", placeholder="e.g. Machine Learning Engineer")
    with gr.Row():
        num_questions = gr.Slider(1, 10, value=5, step=1, label="Number of Questions")
        difficulty = gr.Dropdown(["Easy", "Medium", "Hard"], value="Medium", label="Difficulty")
        qtype = gr.Dropdown(["Technical", "Behavioral", "Mixed"], value="Mixed", label="Question Type")
    with gr.Row():
        temperature = gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature")
        max_tokens = gr.Slider(16, 80, value=64, step=8, label="Max New Tokens per Question")

    with gr.Row():
        generate_btn = gr.Button("Generate", variant="primary")
        clear_btn = gr.Button("Clear")

    output_box = gr.Textbox(label="Generated Interview Questions", lines=10)

    generate_btn.click(
        fn=respond,
        inputs=[role_input, num_questions, difficulty, qtype, temperature, max_tokens],
        outputs=output_box,
    )
    clear_btn.click(
        fn=clear_fields,
        inputs=[],
        outputs=[role_input, num_questions, difficulty, qtype, temperature, max_tokens, output_box],
    )

# Set True only if you want Gradio to create a temporary public share link.
SHARE_PUBLIC_LINK = False
demo.queue().launch(share=SHARE_PUBLIC_LINK, debug=False, prevent_thread_lock=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>